# SolarKG-VN: chạy toàn bộ thực nghiệm trên Kaggle

**Settings cần bật trước khi chạy:** Accelerator = **GPU T4 x2** (1 GPU vẫn chạy được), **Internet = On**.

Thứ tự các bước:
1. Lấy code và cài thư viện
2. Thực nghiệm chính: Bảng 7–10, Hình 1–3
3. Benchmark chuẩn UMLS / Kinship / Nations (R2-7, R2-8)
4. Đo thời gian chạy và khả năng mở rộng (R1-B, R2-4)
5. Baseline LLM và trích xuất bằng LLM với DeepSeek V4.1 Flash qua API (R1-A, R2-6)

Kết quả được copy vào `/kaggle/working/results` sau mỗi bước, nên phiên có bị ngắt giữa chừng thì vẫn giữ được những bước đã xong.
Bước nào chạy rồi thì đặt cờ tương ứng ở cell cấu hình thành `False` để bỏ qua.

In [ ]:
# Cấu hình: bật/tắt từng bước
RUN_MAIN        = True   # ~1 giờ
RUN_BENCHMARKS  = True   # ~1-2 giờ
RUN_SCALABILITY = True   # ~30-60 phút
RUN_LLM         = True   # DeepSeek V4.1 Flash qua API (cần secret DEEPSEEK_API_KEY)
DEEPSEEK_MODEL  = "deepseek-ai/DeepSeek-V4.1-Flash-1017"
DEEPSEEK_BASE_URL = "https://inference.14092005.xyz/openai/v1"
DEEPSEEK_EXTRA_BODY = '{"enable_thinking": false}'   # tắt chế độ thinking
LLM_MAX_DOCS    = 500    # số tài liệu dùng để đánh giá trích xuất

## 1. Lấy code và cài thư viện

In [ ]:
import os, subprocess, shutil, time
if not os.path.exists("/kaggle/working/ai-knowledge-reasoning"):
    !git clone https://github.com/McGarrett7/ai-knowledge-reasoning.git /kaggle/working/ai-knowledge-reasoning
%cd /kaggle/working/ai-knowledge-reasoning
!git pull -q
!pip install -q -r requirements.txt

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
!nvidia-smi --query-gpu=name,memory.total --format=csv || true
!python -m unittest discover -s tests

In [ ]:
OUT = "/kaggle/working/results"
def save():
    shutil.copytree("results", OUT, dirs_exist_ok=True)
    print("saved ->", OUT)

In [ ]:
import torch, subprocess, sys
N_GPU = torch.cuda.device_count()
DEVICES = [f"cuda:{i}" for i in range(N_GPU)] or ["cpu"]
print("devices:", DEVICES)

def parallel(jobs, tag):
    """jobs: list of argument lists for main.py. Job i runs on DEVICES[i % len(DEVICES)];
    jobs on the same device run one after another, different devices run in parallel."""
    groups = {}
    for i, args in enumerate(jobs):
        groups.setdefault(DEVICES[i % len(DEVICES)], []).append(args)
    procs = []
    for dev, arglist in groups.items():
        cmd = " && ".join(f"{sys.executable} main.py " + " ".join(map(str, a)) for a in arglist)
        log = open(f"results/{tag}_{dev.replace(':', '')}.log", "w")
        env = {**os.environ, "KG_DEVICE": dev, "PYTHONIOENCODING": "utf-8"}
        procs.append((dev, subprocess.Popen(cmd, shell=True, env=env, stdout=log, stderr=subprocess.STDOUT)))
        print(f"[{dev}] {cmd}")
    for dev, p in procs:
        p.wait()
        print(f"[{dev}] exit code {p.returncode}")
        !tail -n 5 results/{tag}_{dev.replace(':', '')}.log

## 2. Thực nghiệm chính (5 seed, chia đều cho các GPU): Bảng 7–10, các hình
Mỗi seed được lưu vào `results/shards/seed_<s>.json`, sau đó lệnh `summarize` gộp lại thành bảng và hình.

In [ ]:
if RUN_MAIN:
    !rm -rf results/shards
    seeds = [0, 1, 2, 3, 4]
    # one job per seed, distributed round-robin over the GPUs
    parallel([["run", "--seeds", s, "--no-summary"] for s in seeds], "run_main")
    !python main.py summarize
    save()

## 3. Benchmark chuẩn: UMLS, Kinship, Nations (3 seed, chia dataset cho các GPU)

In [ ]:
if RUN_BENCHMARKS:
    # each dataset writes its own results/benchmark_<name>.json
    parallel([["benchmarks", "--datasets", d] for d in ["kinship", "umls", "nations"]], "run_benchmarks")
    save()

## 4. Đo thời gian chạy và khả năng mở rộng
Bước này cố ý chỉ chạy **một tiến trình trên một GPU**, sau khi các bước trước đã xong, để số đo thời gian không bị ảnh hưởng.

In [ ]:
if RUN_SCALABILITY:
    !KG_DEVICE={DEVICES[0]} python main.py scalability 2>&1 | tee results/run_scalability.log
    save()

## 5. LLM: DeepSeek V4.1 Flash qua API (R1-A, R2-6)
Trước khi chạy, thêm key vào **Add-ons → Secrets**, đặt tên `DEEPSEEK_API_KEY`, rồi bật quyền truy cập cho notebook.
Không dán key trực tiếp vào notebook. Câu trả lời của LLM được cache trong `results/llm_cache/`, nên chạy lại không bị tính phí lần nữa.

In [ ]:
if RUN_LLM:
    from kaggle_secrets import UserSecretsClient
    os.environ["DEEPSEEK_API_KEY"] = UserSecretsClient().get_secret("DEEPSEEK_API_KEY")
    os.environ["DEEPSEEK_BASE_URL"] = DEEPSEEK_BASE_URL
    os.environ["DEEPSEEK_EXTRA_BODY"] = DEEPSEEK_EXTRA_BODY
    # R1-A: zero-shot và KG-RAG, có và không có lớp paraconsistent
    !python main.py llm --backend deepseek --model {DEEPSEEK_MODEL} 2>&1 | tee results/run_llm.log
    # R2-6: chất lượng trích xuất triple bằng LLM
    !python main.py extract --backend deepseek --model {DEEPSEEK_MODEL} --max-docs {LLM_MAX_DOCS} 2>&1 | tee results/run_extract.log
    save()

## 6. Xem nhanh và tải kết quả về

In [ ]:
from IPython.display import Image, Markdown, display
if os.path.exists("results/tables.md"):
    display(Markdown(open("results/tables.md", encoding="utf-8").read()))
for f in sorted(os.listdir("results/figures")) if os.path.exists("results/figures") else []:
    print(f); display(Image(f"results/figures/{f}", width=650))
shutil.make_archive("/kaggle/working/results", "zip", OUT)
print("Tải file /kaggle/working/results.zip ở tab Output")